# Лабораторная работа №9

In [2]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, precision_recall_curve

### 1. Загрузите файл classification.csv. 
##### В нем записаны истинные классы объектов выборки (колонка true) и ответы некоторого классификатора (колонка predicted).

In [3]:
df_class = pd.read_csv('classification.csv')

### 2. Заполните таблицу ошибок классификации. Для этого подсчитайте величины TP, FP, FN и TN согласно их определениям. Например, FP — это количество объектов, имеющих класс 0, но отнесенных алгоритмом к классу 1. 
##### Ответ в данном вопросе — четыре числа через пробел.


In [4]:
# TP: true=1, pred=1
TP = ((df_class['true'] == 1) & (df_class['pred'] == 1)).sum()
# FP: true=0, pred=1
FP = ((df_class['true'] == 0) & (df_class['pred'] == 1)).sum()
# FN: true=1, pred=0
FN = ((df_class['true'] == 1) & (df_class['pred'] == 0)).sum()
# TN: true=0, pred=0
TN = ((df_class['true'] == 0) & (df_class['pred'] == 0)).sum()

print(TP, FP, FN, TN)

43 34 59 64


### 3. Посчитайте основные метрики качества классификатора:
##### • Accuracy (доля верно угаданных) sklearn.metrics.accuracy
##### • Precision (точность) sklearn.metrics.accuracy.precision_score
##### • Recall (полнота) sklearn.metrics.recall_score
##### • F-мера sklearn.metrics.f1_score

In [5]:
y_true = df_class['true']
y_pred = df_class['pred']

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"\nAccuracy (доля верных ответов): {accuracy:.2f}")
print(f"Precision (точность): {precision:.2f}")
print(f"Recall (полнота): {recall:.2f}")
print(f"F-мера: {f1:.2f}")


Accuracy (доля верных ответов): 0.54
Precision (точность): 0.56
Recall (полнота): 0.42
F-мера: 0.48


### 4. Имеется четыре обученных классификатора. В файле scores.csv записаны истинные классы и значения степени принадлежности положительному классу для каждого классификатора на некоторой выборке:
##### • для логистической регрессии вероятность положительного класса (колонка score_logreg),
##### • дляSVM отступотразделяющейповерхности(колонкаscore_svm),
##### • для метрического алгоритма взвешенная сумма классов соседей (колонка score_knn),
##### • для решающего дерева доля положительных объектов в листе (колонка score_tree).
### Загрузите этот файл.


In [6]:
df_scores = pd.read_csv('scores.csv')

### 5. Посчитайте площадь под ROC-кривой для каждого классификатора. 
##### Какой классификатор имеет наибольшее значение метрики AUC-ROC (укажите название столбца с ответами этого классификатора)? Воспользуйтесь функцией sklearn.metrics.roc_auc_score.

In [7]:
y_true_scores = df_scores['true']

# Список колонок с оценками классификаторов
classifiers = ['score_logreg', 'score_svm', 'score_knn', 'score_tree']

# Считаем AUC-ROC для каждого
auc_scores = {}
for clf in classifiers:
    auc = roc_auc_score(y_true_scores, df_scores[clf])
    auc_scores[clf] = auc

# Находим классификатор с наибольшим AUC-ROC
best_auc = max(auc_scores, key=auc_scores.get)
print(f"\nКлассификатор с наибольшим AUC-ROC: {best_auc}")


Классификатор с наибольшим AUC-ROC: score_logreg


### 6. Какой классификатор достигает наибольшей точности (Precision) при полноте (Recall) не менее 70% (укажите название столбца с ответами этого классификатора)? Какое значение точности при этом получается?

In [8]:
best_precision_at_recall = {}

for clf in classifiers:
    # Получаем precision, recall и thresholds
    precision_vals, recall_vals, thresholds = precision_recall_curve(y_true_scores, df_scores[clf])

    # Находим индексы, где recall >= 0.7
    mask = recall_vals >= 0.7
    if mask.any():
        # Берем максимальную precision среди этих индексов
        max_precision = precision_vals[mask].max()
        best_precision_at_recall[clf] = max_precision
    else:
        best_precision_at_recall[clf] = 0

# Находим классификатор с наибольшей Precision
best_clf_precision = max(best_precision_at_recall, key=best_precision_at_recall.get)
best_value = best_precision_at_recall[best_clf_precision]

print(f"Классификатор с наибольшей Precision при Recall >= 70%: {best_clf_precision}")
print(f"Значение Precision: {best_value:.2f}")

Классификатор с наибольшей Precision при Recall >= 70%: score_tree
Значение Precision: 0.65
